# Classificação


In [1]:
%%html
<link rel="stylesheet" href="./style.css">


In [2]:
import pandas as pd
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.preprocessing import FunctionTransformer
from sklearn.tree import DecisionTreeClassifier

from data import (
    df3,
    df3_categorical_features,
    df3_input_features,
    df3_numerical_features,
    df3_target_feature,
)
from plotting_for_task import (
    display_cross_validation_results,
    display_cross_validation_summary,
    display_grid_search_results,
    display_grid_search_summary,
    display_train_test_target_distribution,
)


In [3]:
seeds = [1, 2, 3, 4, 5]
seed = seeds[0]

categorical_features = df3_categorical_features
numerical_features = df3_numerical_features
input_features = df3_input_features
target_feature = df3_target_feature

input_data = df3[input_features].copy()
target_data = df3[target_feature].copy()


In [4]:
def encode_categorical_features(data_frame):
    """Encode categorical features as integer category codes."""
    encoded_data = data_frame.copy()

    for feature in categorical_features:
        encoded_data[feature] = encoded_data[feature].cat.codes

    return encoded_data


categorical_feature_indices = [
    input_features.index(feature) for feature in categorical_features
]


## Separação dos dados

Separamos os dados em **treino** (`80%`) e **teste** (`20%`) de forma estratificada pelo atributo objetivo:

- **Age group**: `10-19`, `20-29`, `30-39`, `40-49`, `50-59`, `60-69`, `70+`.


In [5]:
test_size = 0.2

(
    input_data_for_train,
    input_data_for_test,
    target_data_for_train,
    target_data_for_test,
) = train_test_split(
    input_data,
    target_data,
    test_size=test_size,
    random_state=seed,
    stratify=target_data,
)

In [6]:
display_train_test_target_distribution(
    target_data_for_train,
    target_data_for_test,
);

Age Group,Train N,Test N,Total N,Train %,Test %
10-19,46,12,58,79.3%,20.7%
20-29,158,40,198,79.8%,20.2%
30-39,90,22,112,80.4%,19.6%
40-49,122,31,153,79.7%,20.3%
50-59,162,40,202,80.2%,19.8%
60-69,92,23,115,80.0%,20.0%
70+,69,17,86,80.2%,19.8%
General,739,185,924,80.0%,20.0%


## Decision tree


### Pipeline


In [7]:
def _build_decision_tree_pipeline(
    random_state,
    use_smote,
):
    """Build a Decision Tree pipeline."""

    encoder = (
        "encoder",
        FunctionTransformer(
            encode_categorical_features,
            validate=False,
        ),
    )

    classifier = (
        "classifier",
        DecisionTreeClassifier(
            random_state=random_state,
        ),
    )

    if use_smote:
        smote = (
            "smote",
            SMOTENC(
                random_state=random_state,
                categorical_features=categorical_feature_indices,
            ),
        )

        return Pipeline(
            [
                encoder,
                smote,
                classifier,
            ]
        )

    return Pipeline(
        [
            encoder,
            classifier,
        ]
    )

### Balanceamento


In [8]:
def compare_smote(
    input_data,
    target_data,
    seeds,
    n_splits=5,
):
    """Compare Decision Tree performance with and without SMOTE."""

    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro",
    }

    results = []

    for current_seed in seeds:
        stratified_cross_validation = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=current_seed,
        )

        pipelines = {
            "Without SMOTE": _build_decision_tree_pipeline(
                random_state=current_seed,
                use_smote=False,
            ),
            "With SMOTE": _build_decision_tree_pipeline(
                random_state=current_seed,
                use_smote=True,
            ),
        }

        for strategy, pipeline in pipelines.items():
            cross_validation_results = cross_validate(
                pipeline,
                input_data,
                target_data,
                cv=stratified_cross_validation,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
                error_score="raise",
            )

            results.append(
                {
                    "Seed": current_seed,
                    "Strategy": strategy,
                    "Accuracy": cross_validation_results["test_accuracy"].mean(),
                    "Balanced Accuracy": cross_validation_results[
                        "test_balanced_accuracy"
                    ].mean(),
                    "Macro F1": cross_validation_results["test_f1_macro"].mean(),
                }
            )

    results = pd.DataFrame(results)

    comparison = results.groupby("Strategy").agg(
        {
            "Accuracy": ["mean", "std"],
            "Balanced Accuracy": ["mean", "std"],
            "Macro F1": ["mean", "std"],
        }
    )

    comparison.columns = [f"{metric} {stat}" for metric, stat in comparison.columns]

    comparison = comparison.reset_index()

    metrics = [
        "Accuracy",
        "Balanced Accuracy",
        "Macro F1",
    ]

    for metric in metrics:
        comparison[metric] = (
            comparison[f"{metric} mean"].map(lambda value: f"{value:.3f}")
            + " ± "
            + comparison[f"{metric} std"].map(lambda value: f"{value:.3f}")
        )

    return comparison, results

In [9]:
smote_comparison, smote_results = compare_smote(
    input_data_for_train,
    target_data_for_train,
    seeds,
)

metric_columns = [
    "Accuracy",
    "Balanced Accuracy",
    "Macro F1",
]

display_cross_validation_results(
    smote_results,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Cross-Validation Results",
)

display_cross_validation_summary(
    smote_comparison,
    metric_columns=metric_columns,
    caption="Decision Tree: SMOTE Comparison",
)

Seed,Strategy,Accuracy,Balanced Accuracy,Macro F1
1,Without SMOTE,0.346,0.336,0.324
1,With SMOTE,0.355,0.357,0.342
2,Without SMOTE,0.376,0.367,0.359
2,With SMOTE,0.399,0.400,0.388
3,Without SMOTE,0.394,0.377,0.367
3,With SMOTE,0.405,0.411,0.393
4,Without SMOTE,0.367,0.342,0.333
4,With SMOTE,0.375,0.354,0.346
5,Without SMOTE,0.357,0.353,0.339
5,With SMOTE,0.353,0.374,0.347


Strategy,Accuracy,Balanced Accuracy,Macro F1
With SMOTE,0.377 ± 0.024,0.379 ± 0.025,0.363 ± 0.025
Without SMOTE,0.368 ± 0.018,0.355 ± 0.017,0.345 ± 0.018


In [10]:
decision_tree_pipeline = _build_decision_tree_pipeline(
    random_state=seed,
    use_smote=True,
)

### Ajuste de hiper-parâmetros

Utilizamos o **GridSearch** para encontrarmos os melhores valores de hiper-parâmetros.

- Testamos os seguintes **critérios** de decisão: [`gini`, `entropy`].
- Testamos as seguintes **profundidades** máximas: [`3`, `5`, `7`, `10`, `15`, `None`].
- Testamos as seguintes quantidades mínimas de **amostras** para divisão de nó: [`2`, `5`, `10`].
- Testamos as seguintes quantidades mínimas de **amostras** que um nó folha deve ter: [`1`, `2`, `5`].

Como **critério de seleção**, utilizamos `f1_macro`.

- A métrica `F1` que combina _precision_ e _recall_ para cada classe objetivo.
- O `F1 macro` calcula a média dos valores de F1 das três.

Executamos o GridSearch em `5` **folds** estratificados no conjunto de `treino`.


In [11]:
def _build_decision_tree_grid_search(
    random_state,
    param_grid,
):
    """Build a GridSearchCV for the SMOTENC + Decision Tree pipeline."""

    pipeline = _build_decision_tree_pipeline(
        random_state=random_state,
        use_smote=True,
    )

    stratified_cross_validation = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=random_state,
    )

    scoring = {
        "f1_macro": "f1_macro",
        "balanced_accuracy": "balanced_accuracy",
        "accuracy": "accuracy",
    }

    return GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring=scoring,
        refit="f1_macro",
        cv=stratified_cross_validation,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise",
    )

In [12]:
def run_decision_tree_grid_searches(
    input_data,
    target_data,
    seeds,
    param_grid,
):
    """Run Decision Tree GridSearchCV for multiple CV seeds."""

    grid_searches = {}

    for current_seed in seeds:
        grid_search = _build_decision_tree_grid_search(
            random_state=current_seed,
            param_grid=param_grid,
        )

        grid_search.fit(
            input_data,
            target_data,
        )

        grid_searches[current_seed] = grid_search

    return grid_searches

In [13]:
param_grid_of_decision_tree = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 5],
}

decision_tree_grid_searches = run_decision_tree_grid_searches(
    input_data_for_train,
    target_data_for_train,
    seeds=seeds,
    param_grid=param_grid_of_decision_tree,
)


In [14]:
display_grid_search_results(
    decision_tree_grid_searches,
    caption="Decision Tree: Grid Search Results",
)

Seed,Criterion,Max Depth,Min Samples Leaf,Min Samples Split,F1 Macro,F1 Macro Std,Balanced Accuracy,Balanced Accuracy Std,Accuracy,Accuracy Std
1,entropy,7,1,2,0.3836,0.0198,0.4126,0.0078,0.4046,0.0166
1,entropy,7,2,2,0.3777,0.0251,0.4068,0.0171,0.4032,0.0144
1,entropy,7,1,5,0.3755,0.0246,0.4051,0.0141,0.4005,0.0207
1,gini,7,2,5,0.3745,0.0276,0.4093,0.0193,0.4168,0.0096
1,entropy,7,2,5,0.3734,0.0197,0.4034,0.0136,0.3992,0.0128
1,gini,7,2,2,0.3733,0.0322,0.4075,0.0198,0.4141,0.0125
1,gini,7,1,5,0.3732,0.0267,0.4092,0.0200,0.4127,0.0133
1,gini,7,1,2,0.3721,0.0306,0.4052,0.0204,0.4114,0.0102
1,gini,7,1,10,0.3693,0.0346,0.4057,0.0237,0.4114,0.0129
1,gini,7,2,10,0.3661,0.0348,0.4022,0.0244,0.4100,0.0131


In [15]:
display_grid_search_summary(
    decision_tree_grid_searches,
    caption="Decision Tree: Grid Search Comparison",
)

Criterion,Max Depth,Min Samples Split,Min Samples Leaf,F1 Macro,Balanced Accuracy,Accuracy
gini,10,5,2,0.3724 ± 0.0158,0.4028 ± 0.0244,0.3994 ± 0.0179
gini,10,10,1,0.3705 ± 0.0192,0.3987 ± 0.0245,0.3973 ± 0.0211
gini,10,2,2,0.3698 ± 0.0158,0.4005 ± 0.0242,0.3978 ± 0.0185
entropy,7,2,1,0.3680 ± 0.0122,0.3980 ± 0.0124,0.3946 ± 0.0084
gini,10,10,2,0.3677 ± 0.0229,0.3968 ± 0.0278,0.3946 ± 0.0235
gini,10,5,1,0.3668 ± 0.0114,0.3937 ± 0.0205,0.3932 ± 0.0135
entropy,7,2,2,0.3662 ± 0.0089,0.3975 ± 0.0085,0.3951 ± 0.0066
gini,10,2,1,0.3645 ± 0.0155,0.3916 ± 0.0234,0.3892 ± 0.0186
gini,7,5,1,0.3644 ± 0.0148,0.4019 ± 0.0103,0.3976 ± 0.0146
gini,7,5,2,0.3638 ± 0.0129,0.4015 ± 0.0082,0.3989 ± 0.0149


### Predição
